# Baseline model

Poniżej znajduje się skrypt do trenowania modelu. W katalogu znajduje się również checkpoint do załadowania dla aktualnej wersji. Model jest jeszcze niedopracowany i wymaga badania nad architekturą sieci i datasetem.

Użyte dane: 
1. https://www.kaggle.com/datasets/tristanzhang32/ai-generated-images-vs-real-images - Cały dataset treningowy
2. https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images - po 10000 zdjęć do AI i Naturalnych do podzbioru treningowego i po 3000 do testowego (w celu dostarczenia zdjęć z niską rozdzielczością)

In [1]:
import torch
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"CUDA device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("No CUDA devices available.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

No CUDA devices available.


In [ ]:
import torch
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"CUDA device {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
    device = torch.device("cuda")
    print(f"Using device: {device}")
else:
    print("No CUDA devices available - using CPU")
    device = torch.device("cpu")

PyTorch version: 2.8.0+cpu
CUDA available: False
No CUDA devices available - using CPU


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv("KAGGLE_KEY")

os.environ['KAGGLEHUB_CACHE'] = os.path.join(os.getcwd(), "data")

In [4]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)


c:\Users\szepiet33\Documents\Workspace\ai-detector-model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 105M/105M [00:51<00:00, 2.11MB/s] 

Extracting files...


Path to dataset files: c:\Users\szepiet33\Documents\Workspace\ai-detector-model\notebooks\data\datasets\birdy654\cifake-real-and-ai-generated-synthetic-images\versions\3


In [5]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as dsets
import optuna
import random
import numpy as np

# Define AddGaussianNoise at global scope to avoid pickling issues with multiprocessing
class AddGaussianNoise:
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

def get_augmentation_transforms(trial=None):
    """
    Generate augmentation transforms based on Optuna trial suggestions or default values.
    Includes various augmentation methods: noise, rotation, crop, flip, etc.
    """
    if trial is not None:
        # Optuna hyperparameter optimization
        crop_size = trial.suggest_categorical('crop_size', [96, 112, 128, 144])
        rotation_degrees = trial.suggest_int('rotation_degrees', 5, 30)
        brightness = trial.suggest_float('brightness', 0.1, 0.3)
        contrast = trial.suggest_float('contrast', 0.1, 0.3)
        saturation = trial.suggest_float('saturation', 0.1, 0.3)
        hue = trial.suggest_float('hue', 0.05, 0.15)
        gaussian_noise_std = trial.suggest_float('gaussian_noise_std', 0.01, 0.1)
        use_horizontal_flip = trial.suggest_categorical('use_horizontal_flip', [True, False])
        use_vertical_flip = trial.suggest_categorical('use_vertical_flip', [True, False])
        use_affine = trial.suggest_categorical('use_affine', [True, False])
        use_elastic = trial.suggest_categorical('use_elastic', [True, False])
    else:
        # Default values
        crop_size = 128
        rotation_degrees = 15
        brightness = 0.2
        contrast = 0.2
        saturation = 0.2
        hue = 0.1
        gaussian_noise_std = 0.05
        use_horizontal_flip = True
        use_vertical_flip = False
        use_affine = True
        use_elastic = False

    augmentations = []
    
    # Random resized crop
    augmentations.append(transforms.RandomResizedCrop(crop_size, scale=(0.8, 1.0)))
    
    # Geometric transformations
    if use_horizontal_flip:
        augmentations.append(transforms.RandomHorizontalFlip(p=0.5))
    
    if use_vertical_flip:
        augmentations.append(transforms.RandomVerticalFlip(p=0.3))
    
    # Rotation
    augmentations.append(transforms.RandomRotation(rotation_degrees))
    
    # Affine transformations
    if use_affine:
        augmentations.append(transforms.RandomAffine(
            degrees=0, 
            translate=(0.1, 0.1), 
            scale=(0.9, 1.1),
            shear=5
        ))
    
    # Color jittering
    augmentations.append(transforms.ColorJitter(
        brightness=brightness,
        contrast=contrast,
        saturation=saturation,
        hue=hue
    ))
    
    # Convert to tensor
    augmentations.append(transforms.ToTensor())
    
    # Add Gaussian noise (using global class)
    augmentations.append(AddGaussianNoise(0., gaussian_noise_std))
    
    # Elastic transform (if requested)
    if use_elastic:
        augmentations.append(transforms.ElasticTransform(alpha=50.0, sigma=5.0))
    
    # Normalization (always last)
    augmentations.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    
    return transforms.Compose(augmentations)

# Create transforms - default configuration for now
train_transform = get_augmentation_transforms()

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = dsets.ImageFolder(root=os.path.join(path, "train"), transform=train_transform)
test_dataset = dsets.ImageFolder(root=os.path.join(path, "test"), transform=test_transform)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=8,  # Reduced from 12 to 8 as suggested by PyTorch warning
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=8,  # Reduced from 12 to 8 as suggested by PyTorch warning
    pin_memory=True,
    prefetch_factor=2
)

In [6]:
print(f"Dataloaders: {train_loader, test_loader}") 
print(f"Length of train dataloader: {len(train_loader)} batches of {128}")
print(f"Length of test dataloader: {len(test_loader)} batches of {128}")

Dataloaders: (<torch.utils.data.dataloader.DataLoader object at 0x000001FD2E0292B0>, <torch.utils.data.dataloader.DataLoader object at 0x000001FD2DF86D50>)
Length of train dataloader: 782 batches of 128
Length of test dataloader: 157 batches of 128


In [7]:
from torch import nn
from torch.nn import functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score

class ResidualBlock(nn.Module):
    """
    Two 3×3 convs with batchnorm and ReLU, plus skip connection.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        skip = self.skip(x)
        return self.relu(out + skip)

class CustomBinaryCNN(nn.Module):
    """
    Custom CNN for AI vs. natural image classification.
    - 4 residual convolutional stages
    - SpatialDropout2d for regularization
    - Global average pooling
    - Small classification head
    """
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            ResidualBlock(3, 32),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )
        self.stage2 = nn.Sequential(
            ResidualBlock(32, 64),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )
        self.stage3 = nn.Sequential(
            ResidualBlock(64, 128),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )
        self.stage4 = nn.Sequential(
            ResidualBlock(128, 256),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.stage1(x)    # [B,32,H/2,W/2]
        x = self.stage2(x)    # [B,64,H/4,W/4]
        x = self.stage3(x)    # [B,128,H/8,W/8]
        x = self.stage4(x)    # [B,256,H/16,W/16]
        x = self.global_pool(x)  # [B,256,1,1]
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits
    
    def predict(self, x):
        self.eval()
        with torch.no_grad():
            x = self(x)
            return self.sigmoid(x)
    


In [8]:
from torch import nn
from torch.nn import functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score


def test_model(model, test_loader, loss_fn):
    model.eval()

    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for i, (features, labels) in enumerate(test_loader):
            features, labels = features.to(device), labels.to(device)
            outputs = model.predict(features)
            
            all_labels.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.view(-1).cpu().numpy())

            #batch_accuracy = np.mean((outputs.view(-1).numpy() > 0.5) == labels.numpy())
            #batch_auc = roc_auc_score(labels.numpy(), outputs.view(-1).numpy())
            #print(f"Batch {i+1}/{len(test_loader)}, Loss: {loss_value.item():.4f}, Accuracy: {batch_accuracy:.4f}, AUC: {batch_auc:.4f}")
    
    accuracy = np.mean((np.array(all_outputs) > 0.5) == np.array(all_labels))
    
    auc_score = roc_auc_score(all_labels, all_outputs)
    precision = precision_score(all_labels, (np.array(all_outputs) > 0.5).astype(int))
    recall = recall_score(all_labels, (np.array(all_outputs) > 0.5).astype(int))
    
    print(f"Test Accuracy: {accuracy:.4f}, AUC: {auc_score:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")

In [ ]:
# Trenowanie modelu

loss = nn.BCEWithLogitsLoss()
model = CustomBinaryCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 1

from tqdm.auto import tqdm

for epoch in range(epochs):
    model.train()
    correct = 0
    total = 0
    for features, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss_value = loss(outputs.view(-1), labels.float())
        loss_value.backward()
        optimizer.step()
        preds = (outputs.view(-1) > 0.0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_value.item():.4f}, Accuracy: {accuracy:.4f}")

    if (epoch + 1) % 5 == 0:
        test_model(model, test_loader, loss)
        torch.save(model.state_dict(), f"./model_epoch_{epoch+1}.pth")


Training Epoch 1/1:   0%|          | 0/782 [00:00<?, ?it/s]c:\Users\szepiet33\Documents\Workspace\ai-detector-model\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
import optuna
from sklearn.metrics import roc_auc_score

def objective(trial):
    """
    Optuna objective function to optimize augmentation hyperparameters.
    """
    # Get augmentation transforms with trial suggestions
    train_transform_optimized = get_augmentation_transforms(trial)
    
    # Create new dataset with optimized augmentations
    train_dataset_opt = dsets.ImageFolder(
        root=os.path.join(path, "train"), 
        transform=train_transform_optimized
    )
    
    train_loader_opt = torch.utils.data.DataLoader(
        train_dataset_opt,
        batch_size=128,
        shuffle=True,
        num_workers=8,  # Reduced for optimization
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )
    
    # Initialize model
    model = CustomBinaryCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    
    # Train for fewer epochs during optimization
    num_epochs = 10  # Reduced for faster optimization
    
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, labels) in enumerate(train_loader_opt):
            if batch_idx > 50:  # Limit batches per epoch for speed
                break
                
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss_value = loss_fn(outputs.view(-1), labels.float())
            loss_value.backward()
            optimizer.step()
    
    # Evaluate on a subset of test data
    model.eval()
    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for batch_idx, (features, labels) in enumerate(test_loader):
            if batch_idx > 20:  # Limit test batches for speed
                break
                
            features, labels = features.to(device), labels.to(device)
            outputs = model.predict(features)
            
            all_labels.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.view(-1).cpu().numpy())
    
    # Calculate AUC score as optimization metric
    auc_score = roc_auc_score(all_labels, all_outputs)
    
    return auc_score

# Run Optuna optimization
def optimize_augmentations(n_trials=20):
    """
    Optimize augmentation hyperparameters using Optuna.
    """
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    
    print("Best trial:")
    print(f"  Value: {study.best_trial.value}")
    print("  Params:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    
    return study.best_trial.params

# Uncomment to run optimization (warning: this will take time!)
# best_augmentation_params = optimize_augmentations(n_trials=15)
# print("Best augmentation parameters found:", best_augmentation_params)

In [ ]:
# Example: Train with manually set best augmentation parameters
# (You can replace these with the results from Optuna optimization)

# Example best parameters from optimization (replace with actual results)
best_params_example = {
    'crop_size': 128,
    'rotation_degrees': 20,
    'brightness': 0.25,
    'contrast': 0.2,
    'saturation': 0.15,
    'hue': 0.1,
    'gaussian_noise_std': 0.03,
    'use_horizontal_flip': True,
    'use_vertical_flip': False,
    'use_affine': True,
    'use_elastic': False
}

def create_optimized_transforms(params):
    """Create transforms based on optimized parameters"""
    augmentations = []
    
    # Random resized crop
    augmentations.append(transforms.RandomResizedCrop(params['crop_size'], scale=(0.8, 1.0)))
    
    # Geometric transformations
    if params['use_horizontal_flip']:
        augmentations.append(transforms.RandomHorizontalFlip(p=0.5))
    
    if params['use_vertical_flip']:
        augmentations.append(transforms.RandomVerticalFlip(p=0.3))
    
    # Rotation
    augmentations.append(transforms.RandomRotation(params['rotation_degrees']))
    
    # Affine transformations
    if params['use_affine']:
        augmentations.append(transforms.RandomAffine(
            degrees=0, 
            translate=(0.1, 0.1), 
            scale=(0.9, 1.1),
            shear=5
        ))
    
    # Color jittering
    augmentations.append(transforms.ColorJitter(
        brightness=params['brightness'],
        contrast=params['contrast'],
        saturation=params['saturation'],
        hue=params['hue']
    ))
    
    # Convert to tensor
    augmentations.append(transforms.ToTensor())
    
    # Add Gaussian noise (using global class)
    augmentations.append(AddGaussianNoise(0., params['gaussian_noise_std']))
    
    # Elastic transform
    if params['use_elastic']:
        augmentations.append(transforms.ElasticTransform(alpha=50.0, sigma=5.0))
    
    # Normalization
    augmentations.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    
    return transforms.Compose(augmentations)

# Create optimized train transform
optimized_train_transform = create_optimized_transforms(best_params_example)

# Create new dataset with optimized augmentations
optimized_train_dataset = dsets.ImageFolder(
    root=os.path.join(path, "train"), 
    transform=optimized_train_transform
)

optimized_train_loader = torch.utils.data.DataLoader(
    optimized_train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=8,  # Reduced from 12 to 8 as suggested by PyTorch warning
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print("Optimized augmentation transforms created!")
print(f"Best parameters used: {best_params_example}")

# Uncomment to train with optimized augmentations:
# model_optimized = CustomBinaryCNN().to(device)
# optimizer_opt = torch.optim.Adam(model_optimized.parameters(), lr=0.001, weight_decay=1e-4)
# loss_opt = nn.BCEWithLogitsLoss()
# 
# epochs = 50
# 
# for epoch in range(epochs):
#     model_optimized.train()
#     correct = 0
#     total = 0
#     for features, labels in tqdm(optimized_train_loader, desc=f"Optimized Training Epoch {epoch+1}/{epochs}"):
#         features, labels = features.to(device), labels.to(device)
#         optimizer_opt.zero_grad()
#         outputs = model_optimized(features)
#         loss_value = loss_opt(outputs.view(-1), labels.float())
#         loss_value.backward()
#         optimizer_opt.step()
#         preds = (outputs.view(-1) > 0.0)
#         correct += (preds == labels).sum().item()
#         total += labels.size(0)
#     accuracy = correct / total
#     print(f"Optimized Epoch {epoch+1}/{epochs}, Loss: {loss_value.item():.4f}, Accuracy: {accuracy:.4f}")
# 
#     if (epoch + 1) % 5 == 0:
#         test_model(model_optimized, test_loader, loss_opt)
#         torch.save(model_optimized.state_dict(), f"./optimized_model_epoch_{epoch+1}.pth")

In [ ]:
# Walidacja załadowanego modelu

model = CustomBinaryCNN().to(device)
model.load_state_dict(torch.load("./baseline_model.pth", map_location=torch.device('cuda')))

loss = nn.BCEWithLogitsLoss()
test_model(model, test_loader, loss)

/home/ratattwg/miniconda3/envs/py312/lib/python3.12/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:84.)
  return F.conv2d(input, weight, bias, self.stride,
/home/ratattwg/miniconda3/envs/py312/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ratattwg/miniconda3/envs/py312/lib/python3.12/site-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/ratattwg/miniconda3/envs/py312/lib/python3.12/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ratattwg/miniconda3/envs/py312/lib/python3

Test Accuracy: 0.8327, AUC: 0.9156, Precision: 0.8101, Recall: 0.8696
